# Testes de Cointegracao

Cointegracao e um conceito fundamental em econometria de series temporais. Duas (ou mais) series $I(1)$ sao **cointegradas** se existe uma combinacao linear delas que e $I(0)$ (estacionaria).

Se $y_t \sim I(1)$ e $x_t \sim I(1)$, mas $y_t - \beta x_t \sim I(0)$, entao $y$ e $x$ sao cointegradas com vetor $(1, -\beta)$.

Este notebook cobre tres abordagens para teste de cointegracao:

1. **Engle-Granger (2 passos)**: regressao em nivel + teste ADF nos residuos
2. **Johansen (trace e max-eigenvalue)**: abordagem de sistema multivariado via VECM
3. **Phillips-Ouliaris**: similar ao Engle-Granger, mas com correcao nao-parametrica

---

## Conteudo

1. Conceito e intuicao de cointegracao
2. Geracao de par cointegrado sintetico
3. Teste de Engle-Granger
4. Teste de Johansen (via VECM)
5. Teste de Phillips-Ouliaris
6. Aplicacao: PIB EUA vs PIB Brasil
7. Tabela resumo e interpretacao do rank de cointegracao
8. Exercicios

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox.tests_stat import adf_test, engle_granger_test, phillips_ouliaris_test
from chronobox.models.vecm import VECM

import sys
sys.path.insert(0, '..')
from utils.data_generators import generate_cointegrated_pair, generate_unit_root_process
from utils.plot_helpers import plot_cointegration_residuals

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Intuicao de Cointegracao

Imagine duas series que seguem um "passeio aleatorio" individualmente (cada uma e $I(1)$), mas estao ligadas por uma relacao de equilibrio de longo prazo. Quando se afastam do equilibrio, forcas economicas as trazem de volta.

**Exemplo classico**: preco de commodities em dois mercados - desvios temporarios sao eliminados pela arbitragem.

**Modelo ECM (Error Correction):**

$$\Delta y_t = \alpha (y_{t-1} - \beta x_{t-1}) + \varepsilon_t$$

O termo $\alpha$ e a velocidade de ajuste ao equilibrio. Se $\alpha < 0$, o sistema corrige desvios positivos do equilibrio.

## 2. Geracao de Dados Sinteticos

Vamos gerar:
1. Um **par cointegrado** ($y = 1.5x + \text{erro estacionario}$)
2. Duas series $I(1)$ **nao cointegradas** (independentes)

In [ ]:
# Par cointegrado: y = 1.5*x + erro estacionario
df_coint = generate_cointegrated_pair(n=200, beta=1.5, seed=42, sigma_eq=0.5)

# Duas series I(1) independentes (nao cointegradas)
y_indep1 = generate_unit_root_process(n=200, phi=1.0, seed=10)
y_indep2 = generate_unit_root_process(n=200, phi=1.0, seed=20)

# Visualizacao do par cointegrado
fig = plot_cointegration_residuals(
    pd.Series(df_coint['y'].values, name='y'),
    pd.Series(df_coint['x'].values, name='x'),
    pd.Series(df_coint['equilibrium_error'].values, name='erro'),
    title="Par Cointegrado Sintetico (beta=1.5)"
)
plt.show()

In [ ]:
# Verificar que as series individuais sao I(1)
print("Teste ADF nas series individuais (devem ser I(1)):")
for name, series in [("y (cointegrado)", df_coint['y'].values),
                     ("x (cointegrado)", df_coint['x'].values),
                     ("y_indep1", y_indep1.values),
                     ("y_indep2", y_indep2.values)]:
    r = adf_test(series, regression='c', autolag='AIC')
    print(f"  {name:20s}: ADF stat={r.statistic:.4f}, p={r.pvalue:.4f} {'-> I(0)' if r.reject_at_5pct else '-> I(1)'}")

## 3. Teste de Engle-Granger (2 Passos)

O procedimento de Engle-Granger consiste em:

**Passo 1**: Estimar a regressao de cointegracao por OLS:
$$y_t = \hat{\alpha} + \hat{\beta} x_t + \hat{u}_t$$

**Passo 2**: Testar se os residuos $\hat{u}_t$ sao estacionarios usando ADF:
- Se $\hat{u}_t \sim I(0)$: as series sao cointegradas
- Se $\hat{u}_t \sim I(1)$: as series NAO sao cointegradas

**IMPORTANTE**: Os valores criticos do ADF nos residuos de cointegracao sao **diferentes** dos valores criticos padrao (sao mais negativos), pois os residuos sao estimados, nao observados. O chronobox ja usa os valores criticos corretos de MacKinnon.

In [ ]:
# Engle-Granger no par cointegrado - esperamos REJEITAR H0 (nao cointegracao)
print("=== Engle-Granger: Par Cointegrado ===")
result_eg_coint = engle_granger_test(
    df_coint['y'].values, df_coint['x'].values, trend='c', autolag='AIC'
)
print(result_eg_coint.summary())
print(f"\nDecisao: {'Cointegradas!' if result_eg_coint.reject_at_5pct else 'Nao cointegradas'}")

# Coeficiente estimado (deve ser proximo de beta=1.5)
if 'cointegrating_vector' in result_eg_coint.additional_info:
    print(f"Vetor de cointegracao estimado: {result_eg_coint.additional_info['cointegrating_vector']}")
elif 'beta' in result_eg_coint.additional_info:
    print(f"Beta estimado: {result_eg_coint.additional_info['beta']}")

In [ ]:
# Engle-Granger nas series NAO cointegradas
print("=== Engle-Granger: Series Independentes (NAO cointegradas) ===")
result_eg_indep = engle_granger_test(
    y_indep1.values, y_indep2.values, trend='c', autolag='AIC'
)
print(result_eg_indep.summary())
print(f"\nDecisao: {'Cointegradas!' if result_eg_indep.reject_at_5pct else 'Nao cointegradas (correto!)'}")

## 4. Teste de Johansen (via VECM)

O teste de Johansen e uma abordagem **multivariada** baseada no modelo VECM (Vector Error Correction Model). Diferente do Engle-Granger (bivariado), ele pode testar cointegracao em **sistemas com mais de 2 variaveis** e detectar **multiplos vetores de cointegracao**.

O teste gera duas estatisticas:
- **Trace test**: $H_0: r \leq r_0$ contra $H_1: r > r_0$ (pelo menos $r_0+1$ vetores)
- **Max-eigenvalue test**: $H_0: r = r_0$ contra $H_1: r = r_0 + 1$ (exatamente mais um vetor)

O **rank de cointegracao** ($r$) indica quantas relacoes de equilibrio de longo prazo existem:
- $r = 0$: nenhuma cointegracao
- $0 < r < K$: existem $r$ relacoes de cointegracao (K = numero de variaveis)
- $r = K$: todas as series sao estacionarias

No chronobox, o teste de Johansen esta disponivel como metodo da classe `VECM`.

In [ ]:
# Johansen no par cointegrado
# Construir matriz (T, 2) com as duas series
data_coint = np.column_stack([df_coint['y'].values, df_coint['x'].values])

# Instanciar VECM e rodar teste de Johansen
vecm = VECM(lags=2, deterministic='ci')  # 'ci' = constante dentro do espaco de cointegracao
johansen_result = vecm.johansen_test(data_coint)

print("=== Teste de Johansen: Par Cointegrado ===")
print(johansen_result.summary())
print(f"\nRank estimado (trace):         {johansen_result.rank_trace}")
print(f"Rank estimado (max-eigenvalue): {johansen_result.rank_maxeig}")

In [ ]:
# Johansen nas series NAO cointegradas (espera-se rank=0)
data_indep = np.column_stack([y_indep1.values, y_indep2.values])

vecm_indep = VECM(lags=2, deterministic='ci')
johansen_indep = vecm_indep.johansen_test(data_indep)

print("=== Teste de Johansen: Series Independentes ===")
print(johansen_indep.summary())
print(f"\nRank estimado (trace):         {johansen_indep.rank_trace}")
print(f"Rank estimado (max-eigenvalue): {johansen_indep.rank_maxeig}")

### Interpretacao do Rank de Cointegracao

O procedimento sequencial do teste de Johansen:

1. Teste $H_0: r = 0$ (nenhuma cointegracao). Se rejeitar, prossiga.
2. Teste $H_0: r \leq 1$ (no maximo 1 vetor). Se rejeitar, prossiga.
3. Continue ate nao rejeitar.

O rank selecionado e o primeiro $r_0$ para o qual **nao rejeitamos** $H_0$.

Para um sistema com $K=2$ variaveis:
- $r = 0$: nenhuma relacao de longo prazo (series se movem independentemente)
- $r = 1$: uma relacao de cointegracao (o caso tipico)
- $r = 2$: ambas as series seriam $I(0)$ (nao precisaria de VECM)

## 5. Teste de Phillips-Ouliaris

O teste de Phillips-Ouliaris e similar ao Engle-Granger (residual-based), mas usa a correcao nao-parametrica de Phillips-Perron em vez do ADF nos residuos. Isso o torna mais robusto a autocorrelacao e heterocedasticidade nos residuos.

$$H_0: \text{as series nao sao cointegradas}$$
$$H_1: \text{as series sao cointegradas}$$

In [ ]:
# Phillips-Ouliaris no par cointegrado
print("=== Phillips-Ouliaris: Par Cointegrado ===")
result_po_coint = phillips_ouliaris_test(
    df_coint['y'].values, df_coint['x'].values, trend='c'
)
print(result_po_coint.summary())
print(f"Decisao: {'Cointegradas!' if result_po_coint.reject_at_5pct else 'Nao cointegradas'}")

print("\n")

# Phillips-Ouliaris nas series independentes
print("=== Phillips-Ouliaris: Series Independentes ===")
result_po_indep = phillips_ouliaris_test(
    y_indep1.values, y_indep2.values, trend='c'
)
print(result_po_indep.summary())
print(f"Decisao: {'Cointegradas!' if result_po_indep.reject_at_5pct else 'Nao cointegradas (correto!)'}")

## 6. Aplicacao: PIB EUA vs PIB Brasil

Vamos testar se existe relacao de cointegracao entre os PIBs dos EUA e do Brasil. Para isso, precisamos alinhar as series no mesmo periodo.

In [ ]:
# Carregar dados
gdp_us = pd.read_csv('../data/us_gdp_quarterly.csv', parse_dates=['date'], index_col='date')
gdp_br = pd.read_csv('../data/brazil_gdp.csv', parse_dates=['date'], index_col='date')

# Alinhar pelo periodo comum
common_idx = gdp_us.index.intersection(gdp_br.index)
log_us = gdp_us.loc[common_idx, 'log_gdp'].values
log_br = gdp_br.loc[common_idx, 'log_gdp'].values

print(f"Periodo comum: {common_idx[0].strftime('%Y-%m')} a {common_idx[-1].strftime('%Y-%m')}")
print(f"Observacoes: {len(common_idx)}")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(common_idx, log_us, label='Log PIB EUA', color='steelblue')
axes[0].plot(common_idx, log_br, label='Log PIB Brasil', color='coral')
axes[0].legend()
axes[0].set_title('Log PIB em Nivel')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(log_us, log_br, alpha=0.5, s=15, color='steelblue')
z = np.polyfit(log_us, log_br, 1)
axes[1].plot(np.sort(log_us), np.poly1d(z)(np.sort(log_us)), 'r-',
             label=f'BR = {z[0]:.2f}*US + {z[1]:.2f}')
axes[1].set_xlabel('Log PIB EUA')
axes[1].set_ylabel('Log PIB Brasil')
axes[1].set_title('Relacao PIB EUA x Brasil')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Testes de cointegracao: PIB EUA vs Brasil
print("=" * 70)
print("  Testes de Cointegracao: PIB EUA vs Brasil")
print("=" * 70)

# Engle-Granger
print("\n--- Engle-Granger ---")
eg_pib = engle_granger_test(log_br, log_us, trend='c', autolag='AIC')
print(f"  Estatistica: {eg_pib.statistic:.4f}")
print(f"  P-valor:     {eg_pib.pvalue:.4f}")
print(f"  Cointegradas: {'Sim' if eg_pib.reject_at_5pct else 'Nao'}")

# Phillips-Ouliaris
print("\n--- Phillips-Ouliaris ---")
po_pib = phillips_ouliaris_test(log_br, log_us, trend='c')
print(f"  Estatistica: {po_pib.statistic:.4f}")
print(f"  P-valor:     {po_pib.pvalue:.4f}")
print(f"  Cointegradas: {'Sim' if po_pib.reject_at_5pct else 'Nao'}")

# Johansen
print("\n--- Johansen ---")
data_pib = np.column_stack([log_br, log_us])
vecm_pib = VECM(lags=4, deterministic='ci')
joh_pib = vecm_pib.johansen_test(data_pib)
print(joh_pib.summary())

## 7. Tabela Resumo de Resultados

In [ ]:
# Tabela resumo de todos os testes de cointegracao
summary_data = []

# Par cointegrado sintetico
for test_name, result in [("Engle-Granger", result_eg_coint),
                           ("Phillips-Ouliaris", result_po_coint)]:
    summary_data.append({
        'Par': 'Cointegrado sintetico',
        'Teste': test_name,
        'Estatistica': f"{result.statistic:.4f}",
        'P-valor': f"{result.pvalue:.4f}" if result.pvalue is not None else "N/A",
        'Decisao': 'Cointegradas' if result.reject_at_5pct else 'Nao coint.'
    })
summary_data.append({
    'Par': 'Cointegrado sintetico',
    'Teste': 'Johansen (trace)',
    'Estatistica': f"{johansen_result.trace_stat[0]:.4f}",
    'P-valor': '-',
    'Decisao': f'rank={johansen_result.rank_trace}'
})

# Series independentes
for test_name, result in [("Engle-Granger", result_eg_indep),
                           ("Phillips-Ouliaris", result_po_indep)]:
    summary_data.append({
        'Par': 'Independentes',
        'Teste': test_name,
        'Estatistica': f"{result.statistic:.4f}",
        'P-valor': f"{result.pvalue:.4f}" if result.pvalue is not None else "N/A",
        'Decisao': 'Cointegradas' if result.reject_at_5pct else 'Nao coint.'
    })
summary_data.append({
    'Par': 'Independentes',
    'Teste': 'Johansen (trace)',
    'Estatistica': f"{johansen_indep.trace_stat[0]:.4f}",
    'P-valor': '-',
    'Decisao': f'rank={johansen_indep.rank_trace}'
})

# PIB EUA vs Brasil
for test_name, result in [("Engle-Granger", eg_pib),
                           ("Phillips-Ouliaris", po_pib)]:
    summary_data.append({
        'Par': 'PIB EUA x Brasil',
        'Teste': test_name,
        'Estatistica': f"{result.statistic:.4f}",
        'P-valor': f"{result.pvalue:.4f}" if result.pvalue is not None else "N/A",
        'Decisao': 'Cointegradas' if result.reject_at_5pct else 'Nao coint.'
    })
summary_data.append({
    'Par': 'PIB EUA x Brasil',
    'Teste': 'Johansen (trace)',
    'Estatistica': f"{joh_pib.trace_stat[0]:.4f}",
    'P-valor': '-',
    'Decisao': f'rank={joh_pib.rank_trace}'
})

df_summary = pd.DataFrame(summary_data)
print("=" * 80)
print("  TABELA RESUMO: Testes de Cointegracao")
print("=" * 80)
print(df_summary.to_string(index=False))

## 8. Johansen com Diferentes Especificacoes Deterministicas

A especificacao dos termos deterministicos no teste de Johansen afeta os valores criticos. As opcoes no chronobox sao:

| Codigo | Descricao |
|--------|-----------|
| `'nc'` | Sem constante, sem tendencia |
| `'ci'` | Constante restrita ao espaco de cointegracao |
| `'co'` | Constante irrestrita (fora do EC) |
| `'li'` | Tendencia linear restrita ao espaco de cointegracao |
| `'lo'` | Tendencia linear irrestrita |

A escolha mais comum e `'ci'` (constante no espaco de cointegracao).

In [ ]:
# Comparar especificacoes deterministicas no Johansen
print("=" * 70)
print("  Sensibilidade do Johansen a Especificacao Deterministica")
print("  (Par cointegrado sintetico)")
print("=" * 70)

for det in ['nc', 'ci', 'co']:
    vecm_det = VECM(lags=2, deterministic=det)
    joh = vecm_det.johansen_test(data_coint)
    print(f"\n--- deterministic='{det}' ---")
    print(f"  Trace stats:     {[f'{s:.2f}' for s in joh.trace_stat]}")
    print(f"  Trace CVs (95%): {[f'{c:.2f}' for c in joh.trace_crit[:, 1]]}")
    print(f"  Rank (trace):    {joh.rank_trace}")
    print(f"  Rank (max-eig):  {joh.rank_maxeig}")

## Exercicio

### Exercicio 1: Cointegracao com Diferentes Betas

Gere pares cointegrados com diferentes valores de $\beta$ (0.5, 1.0, 2.0, 3.0) e diferentes niveis de ruido no erro de equilibrio ($\sigma_{eq}$ = 0.1, 0.5, 1.0, 2.0):
1. Aplique Engle-Granger e Johansen em cada caso
2. Como a forca da cointegracao (razao sinal/ruido) afeta a deteccao?
3. Monte uma tabela com os resultados

**Dica**: Use `generate_cointegrated_pair(n=200, beta=b, seed=42, sigma_eq=s)`

In [ ]:
# Exercicio 1 - Seu codigo aqui
# Gere pares cointegrados com diferentes betas e sigma_eq


### Exercicio 2: Sistema Trivariado com Johansen

Gere um sistema com 3 variaveis onde:
- $x_1$ e $x_2$ sao cointegradas ($x_1 = 2x_2 + \text{erro}$)
- $x_3$ e um passeio aleatorio independente

Aplique o teste de Johansen com lags=2 e `deterministic='ci'`. O teste detecta corretamente rank=1?

**Dica**: Use `generate_cointegrated_pair` para o par e `generate_unit_root_process` para $x_3$, depois empilhe com `np.column_stack`

In [ ]:
# Exercicio 2 - Seu codigo aqui
# Construa o sistema trivariado e aplique Johansen
